# Step 2 — Train Random Forest crop classifier with fixed hyperparameters

**Input:** merged table from `data/prepared/`  
**What it does:**
- filters to crop-only Level-2 classes
- selects VV/VH time-window features
- trains a Random Forest with fixed hyperparameters
- evaluates on a stratified validation split
- saves the trained model + feature list into `models/`

In [1]:
import json
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_PREP = REPO_ROOT / "data" / "prepared"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Filenames (edit ONLY these if your filenames differ)
data_file = DATA_PREP / "S1_point_all_10d_10m_20180101-20180731_Stratum1_VV-VH.csv"
classes_file = DATA_PREP / "LUCAS_2018_Copernicus_attributes_cropmap_level1-2_FROM_EXPORTS.csv"

# Friendly errors if files are missing
assert data_file.exists(), f"Missing dataset file: {data_file}"
assert classes_file.exists(), f"Missing classes file: {classes_file}"

print("Using data_file:", data_file)
print("Using classes_file:", classes_file)
print("Saving outputs to:", MODELS_DIR)

Using data_file: /workspace/eucropmap-reprod-kz/data/prepared/S1_point_all_10d_10m_20180101-20180731_Stratum1_VV-VH.csv
Using classes_file: /workspace/eucropmap-reprod-kz/data/prepared/LUCAS_2018_Copernicus_attributes_cropmap_level1-2_FROM_EXPORTS.csv
Saving outputs to: /workspace/eucropmap-reprod-kz/models


In [2]:
class_table = pd.read_csv(classes_file)
print(class_table)


       POINT_ID  stratum  LC1   LU1  level_1  level_2
0      47242864        1  B11  U111      200      211
1      47322804        1  B11  U111      200      211
2      47602810        1  B11  U111      200      211
3      47622814        1  B11  U111      200      211
4      47642818        1  B11  U111      200      211
...         ...      ...  ...   ...      ...      ...
30929  36503184        1  E20  U111      500      500
30930  36643156        1  E20  U111      500      500
30931  36703150        1  E20  U111      500      500
30932  36503170        1  F40  U111      200      290
30933  36883144        1  F40  U112      200      290

[30934 rows x 6 columns]


In [3]:
# Load class legend table 

classes_L1 = class_table["level_1"].dropna().unique().tolist()
classes_L2 = class_table["level_2"].dropna().unique().tolist()

df = pd.read_csv(data_file, dtype={'level_1': int, 'level_2': int})
print(f"Loaded dataset with shape {df.shape}")


Loaded dataset with shape (1743815, 46)


In [4]:
#official Level-2 set:
L2_official = {211,212,213,214,215,216,217,218,219,221,222,223,230,231,232,233,240,250,290}



# sremove non crop labled data
bad_L1 = {100,300,500,600}
classes_L2 = [c for c in classes_L2 if c in L2_official]

print(f"Classes in level_1: {classes_L1}")
print(f"Classes in level_2: {classes_L2}")
print(df.head())

Classes in level_1: [200, 500, 300, 600, 100]
Classes in level_2: [211, 212, 213, 214, 215, 216, 218, 219, 221, 222, 223, 231, 232, 233, 230, 240, 250, 290]
   POINT_ID  stratum  level_1  level_2  VH_20180101  VH_20180111  VH_20180121  \
0  47242864        1      200      211   -17.729420   -20.325294   -19.684908   
1  47242864        1      200      211   -17.629759   -20.395664   -19.362911   
2  47322804        1      200      211   -16.761300   -16.439291   -19.003990   
3  47322804        1      200      211   -16.949911   -17.447950   -18.359556   
4  47322804        1      200      211   -16.443756   -16.525919   -18.132175   

   VH_20180131  VH_20180210  VH_20180220  ...  VV_20180421  VV_20180501  \
0   -20.850082   -20.764990   -23.271540  ...   -15.297538   -14.691077   
1   -20.440153   -21.169271   -23.260570  ...   -13.002155   -14.203595   
2   -20.409580   -22.091795   -21.210240  ...   -16.568722   -17.411484   
3   -20.564657   -23.349674   -21.620611  ...   -16.0513

In [5]:
df['Classif'] = df['level_2']  # working label for classification (detailed crop types)
if classes_L2:
    df = df[df['Classif'].isin(classes_L2)]
print(f"Data after filtering to crop classes: {df.shape}")



Data after filtering to crop classes: (604610, 47)


In [6]:

feature_regex = r'(((?<![\w\d])VH_)|((?<![\w\d])VV_))'  
feature_regex += r'(2018(0[1-7]))'  

X = df.filter(regex=feature_regex)
y = df['Classif']

print(f"Selected features matrix X shape: {X.shape}")
print(f"Selected target vector y shape: {y.shape}")
# Check a quick summary of class distribution
print("Class distribution in y:")
print(y.value_counts())
# feature names used in training
feature_names = list(X.columns)
feat_path = MODELS_DIR / "RF_feature_names.json"
with open(feat_path, "w") as f:
    json.dump(feature_names, f, indent=2)
print(f"Saved training feature list to {feat_path}")


Selected features matrix X shape: (604610, 42)
Selected target vector y shape: (604610,)
Class distribution in y:
Classif
211    195821
213     89215
216     69089
232     53663
214     31153
290     24290
215     24001
250     23366
222     22901
240     18351
218     16124
221     14620
230      6360
212      4711
223      3780
219      3446
231      1975
233      1744
Name: count, dtype: int64
Saved training feature list to /workspace/eucropmap-reprod-kz/models/RF_feature_names.json


In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

rfc = RandomForestClassifier(n_jobs=-1)
print("Training Random Forest with fixed hyperparameters on the training split...")
rfc.fit(X_train, y_train)
print("Training complete.")
print("Hyperparameters used:")
print(rfc.get_params())


Training Random Forest with fixed hyperparameters on the training split...
Training complete.
Hyperparameters used:
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False}


In [8]:
y_pred = rfc.predict(X_val)
print("Accuracy on validation set: {:.2f}%".format(100 * (y_pred == y_val).mean()))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))


Accuracy on validation set: 87.02%
Confusion Matrix:
 [[47330     0   529   167    13   591     2     0     9    25     0     0
      0   191     0    28    62     8]
 [  642   427    35    34     0    33     1     0     2     0     0     0
      0     0     0     3     1     0]
 [ 2416     0 19111   141    16   474     4     0     4     8     0     0
      0    50     0    12    64     4]
 [ 1147     0   212  6005    18   298     5     0     2     2     0     0
      0    44     0     7    48     0]
 [ 1089     0   190   155  4298   177     4     0     2     2     0     1
      0    20     0     8    53     1]
 [  771     0   125   122     9 16021     1     0    17    50     0     2
      1    94     0    16    43     0]
 [ 1166     0   141   288     9   173  2209     0     2     2     0     0
      0    15     0     6    20     0]
 [  132     0    10    15     7   142     0   513     3     2     0     0
      0    27     0     5     5     0]
 [   25     0     7     7     1   201     